# NB06: Saliency-Matching Distillation (ablation)

Does forcing the student to *look* where the teacher looks improve the gate?

Standard KD constrains what the student predicts. This ablation adds a term that aligns the
student's Grad-CAM with Teacher A's, which is the property Stage A of the gate relies on
(it scores the fraction of saliency mass falling inside the QRS complex).

This notebook has three parts:

1. **Mechanism check** — verifies the differentiable Grad-CAM actually trains the student.
   Runs standalone in seconds, no checkpoints needed.
2. **Ablation table** — compares the baseline student against the saliency-matched student
   using the JSON artifacts from NB03/NB04/NB05. Needs both arms to have been run.
3. **Stage-A supplementary stats** — reports the fitted logistic coefficients, within-Stage-A
   feature ablations, and paired-bootstrap policy comparisons for the manuscript (§3.6, §4.4).
   Needs baseline NB04/NB05 artifacts.

On a **full run** (`XFAD_DRY_RUN='0'`), NB02–NB05 automatically produce both arms
(baseline + saliency). Saliency artifacts use a `_saliency` suffix so nothing is overwritten.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## RUN MODE — edit the cell below (only switch in this notebook)

| `XFAD_DRY_RUN` | Meaning |
|---|---|
| `'1'` | **Smoke test** — tiny samples, 1 epoch, fast sanity check (~minutes) |
| `'0'` | **Full paper run** — all data, full training, both student arms (~hours) |

On a **full run**, NB02–NB05 automatically process **baseline** and **saliency** students (everything NB06 needs).

Then: **Runtime → Run all** (restart session after changing the switch).


In [ ]:
import os

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  ONLY SWITCH IN THIS NOTEBOOK                                        ║
# ║  '0' = FULL PAPER RUN (all data, both student arms)                  ║
# ║  '1' = SMOKE TEST (tiny data, baseline student only)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
os.environ['XFAD_DRY_RUN'] = '0'


In [ ]:
import os
import sys
from pathlib import Path

def bootstrap_xfad_root() -> Path:
    """Locate repo root and put it on sys.path (works in Colab + local)."""
    candidates = []
    env = os.environ.get('XFAD_ROOT')
    if env:
        candidates.append(Path(env))
    candidates.extend([
        Path('/content/drive/MyDrive/XFAD'),
        Path.cwd(),
        Path.cwd().parent,
    ])
    candidates.extend(Path.cwd().resolve().parents)
    seen = set()
    for p in candidates:
        try:
            p = p.resolve()
        except OSError:
            continue
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'config.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            os.environ.setdefault('XFAD_ROOT', str(p))
            return p
    raise FileNotFoundError(
        'XFAD repo root not found (expected src/config.py). '
        'In Colab: mount Drive first, then either keep the project at '
        '/content/drive/MyDrive/XFAD or set '
        "os.environ['XFAD_ROOT'] = '/content/drive/MyDrive/<your>/XFAD'."
    )

REPO_ROOT = bootstrap_xfad_root()
print('REPO_ROOT:', REPO_ROOT)

from src.utils.io import set_seed
set_seed(42)

from src.utils.io import resolve_dry_run, print_run_banner
DRY_RUN = resolve_dry_run()
import torch
from src.config import KD_WEIGHT_SALIENCY, MODEL_DIR, RESULTS_DIR
from src.explain.gradcam import grad_cam_1d_batch
from src.models.saliency_kd import saliency_kd_term
from src.models.student import MultiTaskStudent
from src.models.teacher_a import TeacherA
device = torch.device('cuda' if torch.cuda.is_available() and not DRY_RUN else 'cpu')
print('Saliency KD weight:', KD_WEIGHT_SALIENCY)

from src.utils.io import print_run_banner
print_run_banner(notebook='NB06_Stretch_Saliency', dry_run=DRY_RUN)


In [ ]:
# Part 1: mechanism check - the alignment loss must be differentiable end to end.
student = MultiTaskStudent().to(device)
teacher = TeacherA().to(device)
teacher_ckpt = os.path.join(MODEL_DIR, 'teacher_a.pt')
if os.path.exists(teacher_ckpt):
    teacher.load_state_dict(torch.load(teacher_ckpt, map_location=device))
    print('Loaded trained Teacher A')
else:
    print('Teacher A checkpoint missing - using a random-init teacher for the mechanism check only')
teacher.eval()

x = torch.randn(8, 2, 250, device=device)
targets = torch.randint(0, 5, (8,), device=device)

# Without create_graph the head only sees the CAM through detached gradient weights,
# so the term silently fails to train the part of the network that decides where to look.
cam_detached = grad_cam_1d_batch(student, x, targets, MultiTaskStudent.LAST_CONV, task='A')
g_detached = torch.autograd.grad(cam_detached.pow(2).mean(), student.head_a.weight, allow_unused=True)[0]
cam_diff = grad_cam_1d_batch(student, x, targets, MultiTaskStudent.LAST_CONV, task='A', create_graph=True)
g_diff = torch.autograd.grad(cam_diff.pow(2).mean(), student.head_a.weight, allow_unused=True)[0]
print(f'head_a gradient  create_graph=False: {g_detached is not None}  create_graph=True: {g_diff is not None}')
assert g_detached is None and g_diff is not None, 'differentiable Grad-CAM is not wired up'

# eval() keeps BatchNorm from injecting batch noise into the CAM during this short demo.
student.eval()
optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)
losses = []
for step in range(10):
    loss, s_cam, t_cam = saliency_kd_term(
        student, teacher, x, targets,
        student_layer=MultiTaskStudent.LAST_CONV,
        teacher_layer=TeacherA.LAST_CONV,
    )
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(float(loss))
print(f'Alignment loss over 10 steps: {losses[0]:.4f} -> {losses[-1]:.4f}')
assert losses[-1] < losses[0], 'alignment loss did not decrease - check the Grad-CAM graph'
print('Mechanism check passed: the saliency term trains the student.')


## Part 2: ablation table

Compares the two arms on the metrics the paper claims: held-out accuracy and macro-F1,
saliency mass inside the QRS, deletion AUC, and Stage A+B false-alarm reduction.

If the saliency arm is missing, the cell below prints exactly what to run.

In [ ]:
from src.paper.tables import ablation_table, load_variant_results
from src.utils.io import student_checkpoint_path

baseline = load_variant_results(RESULTS_DIR, 'baseline')
saliency = load_variant_results(RESULTS_DIR, 'saliency')

for name, arm in (('baseline', baseline), ('saliency', saliency)):
    have = sorted(k for k, v in arm.items() if v)
    print(f'{name:9s} artifacts present: {have or "none"}')

if not any(saliency.values()):
    ckpt = student_checkpoint_path('saliency')
    print(f"""
The saliency arm has not been run yet. Run NB02 → NB03 → NB04 → NB05 with XFAD_DRY_RUN='0'. They automatically train and
evaluate both student arms. NB02 writes {ckpt}; NB03/04/05 write *_saliency.json files.
Re-run this notebook afterwards.""")
elif not any(baseline.values()):

    print('\n'+'Baseline artifacts missing: run NB02-NB05 with XFAD_DRY_RUN="0" first')
else:
    table = ablation_table(baseline, saliency)
    print('\n' + table)
    ablation_path = os.path.join(RESULTS_DIR, 'ablation_saliency.md')
    with open(ablation_path, 'w', encoding='utf-8') as f:
        f.write(table + '\n')
    print('\nSaved', ablation_path)


## Part 3: Stage-A coefficients and paired bootstrap tests

Reports the fitted Stage-A logistic-regression coefficients, validation feature ablations,
and paired bootstrap FA-reduction differences between gate policies on DS2 test.

Requires baseline artifacts from NB04 (`gate_model.joblib`, `gate_thresholds.json`) and
NB05 (`gate_eval_ds2.json`). On a smoke test, uses a small alarm subsample and 100
bootstrap resamples for speed.

In [ ]:
import json
import os

from src.gate.supplementary import format_supplementary_summary, run_gate_supplementary_analysis
from src.utils.io import variant_result_path

gate_model_path = variant_result_path('gate_model', 'baseline', ext='.joblib')
gate_eval_path = variant_result_path('gate_eval_ds2', 'baseline')
missing = [p for p in (gate_model_path, gate_eval_path) if not os.path.exists(p)]

if missing:
    print('Baseline gate artifacts missing:')
    for p in missing:
        print(' -', p)
    print("Run NB04 and NB05 with XFAD_DRY_RUN='0' on the baseline student first.")
else:
    n_boot = 100 if DRY_RUN else 1000
    max_alarms = 200 if DRY_RUN else 0
    print(f'Running Stage-A supplementary analysis (n_boot={n_boot}, max_alarms={max_alarms or "all"})')

    payload = run_gate_supplementary_analysis(
        device=device,
        student_variant='baseline',
        n_boot=n_boot,
        max_alarms=max_alarms,
        progress=True,
        results_dir=RESULTS_DIR,
        save=True,
    )

    print(format_supplementary_summary(payload))
    print('\nSaved', payload.get('output_path', os.path.join(RESULTS_DIR, 'gate_supplementary.json')))
    print('\nJSON preview:')
    print(json.dumps({
        'coefficients': payload['coefficients'],
        'stage_a_ablations_validation': payload['stage_a_ablations_validation'],
        'paired_bootstrap_test': payload['paired_bootstrap_test'],
    }, indent=2))